In [1]:
import os
import io
from PIL import Image
import pandas as pd
import numpy as np


class _DummyConfig:
    @staticmethod
    def list_physical_devices(_):
        return []  # No GPUs detected in dummy mode

    class Experimental:
        @staticmethod
        def enable_op_determinism():
            pass

        @staticmethod
        def set_experimental_options(_):
            pass

    experimental = Experimental


class _DummyKerasUtils:
    @staticmethod
    def set_random_seed(_):
        pass


class _DummyRandom:
    @staticmethod
    def set_seed(_):
        pass


class _DummyDistribution:
    class OneDeviceStrategy:
        def __init__(self, device):
            pass

    class MirroredStrategy:
        def __init__(self):
            pass


class _DummyTF:
    __version__ = "dummy"
    config = _DummyConfig()
    keras = type("keras", (), {"utils": _DummyKerasUtils()})
    random = _DummyRandom()
    distribute = _DummyDistribution()


tf = _DummyTF()
tf_available = True


def reset_default_graph():
    """Placeholder for tensorflow.python.framework.ops.reset_default_graph."""
    pass


print("TensorFlow version =", getattr(tf, "__version__", "dummy"))

gpus = tf.config.list_physical_devices("GPU") if hasattr(tf, "config") else []
if len(gpus) <= 1:
    strategy = (
        tf.distribute.OneDeviceStrategy(device="/gpu:0")
        if hasattr(tf, "distribute")
        else None
    )
    print(f"Using {len(gpus)} GPU (dummy mode)")
else:
    strategy = tf.distribute.MirroredStrategy() if hasattr(tf, "distribute") else None
    print(f"Using {len(gpus)} GPUs (dummy mode)")

print("EfficientNet import skipped (baseline model)")

PLATFORM = "kaggle"  # local kaggle
NEEDTRAIN = False
DATATYPE = ["eeg", "spe", "img"]  # 'eeg', 'spe', 'img', 'stft'
STAGETRAIN = [2, 3]
STAGETEST = 3
print(DATATYPE)
LOAD_MODELS_FROM = "models2024030301"
if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

EEG_LENGTH = 30  # s
SFREQ = 100

HIGH = 128  # 128
LENGTH = 256  # 256

IMG_HIGH = 64
IMG_WIDE = 256

SEED = 2024
BATCHSIZE = 16

READ_SPEC_FILES = False
READ_EEG_FILES = False
READ_IMG_FILES = False
READ_STFT_FILES = False

filter_range = [0.5, 40]

BRAIN = {
    "LL": ["Fp1-F7", "F7-T3", "T3-T5", "T5-O1"],
    "RL": ["Fp2-F8", "F8-T4", "T4-T6", "T6-O2"],
    "LP": ["Fp1-F3", "F3-C3", "C3-P3", "P3-O1"],
    "RP": ["Fp2-F4", "F4-C4", "C4-P4", "P4-O2"],
}

np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
if hasattr(tf, "random"):
    tf.random.set_seed(SEED)
if hasattr(tf.keras.utils, "set_random_seed"):
    tf.keras.utils.set_random_seed(SEED)
if hasattr(tf.config.experimental, "enable_op_determinism"):
    tf.config.experimental.enable_op_determinism()

MIX = True
if (
    MIX
    and hasattr(tf.config, "optimizer")
    and hasattr(tf.config.optimizer, "set_experimental_options")
):
    tf.config.optimizer.set_experimental_options({"auto_mixed_precision": True})
    print("Mixed precision enabled")
else:
    print("Using full precision")

if PLATFORM == "local":
    train_path = "./input/hms-harmful-brain-activity-classification/train.csv"
else:
    train_path = "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"

df = pd.read_csv(train_path)
TARGETS = df.columns[-6:]  # last six columns are the vote columns
print("Train shape:", df.shape)
print("Targets", list(TARGETS))

votes = df[TARGETS].values.astype(float)
row_sums = votes.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
probabilities = votes / row_sums
baseline_probs = probabilities.mean(axis=0)  # shape (6,)

print("Baseline class probabilities:", baseline_probs)

# Reduced pseudo‑count to lessen over‑smoothing
alpha = 1.0  # smaller pseudo‑count for each class
patient_group_raw = df.groupby("patient_id")[list(TARGETS)].sum()
patient_group = patient_group_raw + alpha  # add pseudo‑counts

patient_vote_counts = patient_group_raw.sum(axis=1).values  # shape (n_patients,)

patient_row_sums = patient_group.sum(axis=1).values[:, None]
patient_row_sums[patient_row_sums == 0] = 1.0
patient_probs = patient_group / patient_row_sums  # DataFrame indexed by patient_id

if PLATFORM == "local":
    test_path = "./input/hms-harmful-brain-activity-classification/test.csv"
else:
    test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"

test = pd.read_csv(test_path)
print("Test shape:", test.shape)

sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
test_patients = (
    test["patient_id"]
    if "patient_id" in test.columns
    else pd.Series([np.nan] * len(test))
)

# Reduced blending regularisation to allow patient‑specific probabilities to influence predictions
beta = 1.0  # smaller blending regularisation
prob_list = []
for pid in test_patients:
    if pid in patient_probs.index:
        patient_prob = patient_probs.loc[pid].values
        n = patient_group_raw.loc[pid].sum()  # total raw votes for this patient
        weight = n / (n + beta)  # now gives substantial weight to patient data
        blended = weight * patient_prob + (1 - weight) * baseline_probs
        prob_list.append(blended)
    else:
        prob_list.append(baseline_probs)

prob_df = pd.DataFrame(prob_list, columns=TARGETS)
sub = pd.concat([sub, prob_df], axis=1)

sub[TARGETS] = sub[TARGETS].div(sub[TARGETS].sum(axis=1), axis=0)

submission_path = "submission.csv"
sub.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
print("Submission shape:", sub.shape)
print("Row sums (first 5):", sub[TARGETS].sum(axis=1).head().values)

TensorFlow version = dummy
Using 0 GPU (dummy mode)
EfficientNet import skipped (baseline model)
['eeg', 'spe', 'img']
Using full precision
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Baseline class probabilities: [0.20876764 0.1314852  0.12428287 0.14232007 0.18053678 0.21260744]
Test shape: (9850, 3)
Submission written to submission.csv
Submission shape: (9850, 7)
Row sums (first 5): [1. 1. 1. 1. 1.]
